In [ ]:
import math
import numpy as np
from scipy.ndimage import gaussian_filter

import rasterio
from rasterio.windows import Window

def blur(image: np.ndarray, psf_sigma: float, source_gsd: float, target_gsd: float) -> np.ndarray:
    """Apply Gaussian PSF blur scaled to the GSD ratio."""
    scale_factor = target_gsd / source_gsd
    sigma_pixels = psf_sigma * scale_factor
    blurred = np.empty_like(image, dtype=np.float64)
    for c in range(image.shape[0]):
        blurred[c] = gaussian_filter(image[c].astype(np.float64), sigma=sigma_pixels)
    return blurred
 
 
def noise(image: np.ndarray, snr_db: float) -> np.ndarray:
    """Add Poisson noise at a given SNR (dB)."""
    snr_linear = 10 ** (snr_db / 20)
    signal_power = np.mean(image.astype(np.float64) ** 2)
    noise_std = math.sqrt(signal_power) / snr_linear
    scale = 1.0 / (noise_std ** 2 + 1e-10)
    noisy = np.random.poisson(np.clip(image.astype(np.float64) * scale, 0, None))
    return noisy / scale
 
 
def downsample(image: np.ndarray, source_gsd: float, target_gsd: float) -> np.ndarray:
    """Average-pool downsample by the GSD ratio (integer scale)."""
    scale = int(round(target_gsd / source_gsd))
    if scale == 1:
        return image
    C, H, W = image.shape
    new_H = H // scale
    new_W = W // scale
    cropped = image[:, : new_H * scale, : new_W * scale]
    return cropped.reshape(C, new_H, scale, new_W, scale).mean(axis=(2, 4))


def extract_tile(source_image: str, min_x: float, max_x: float, 
                 min_y: float, max_y: float) -> tuple[np.ndarray, any]:
    """Extract a tile from the source image using rasterio."""
    with rasterio.open(source_image) as src:
        window = rasterio.windows.from_bounds(min_x, min_y, max_x, max_y, src.transform)
        tile = src.read(window=window)
        transform = src.window_transform(window)
    return tile, transform

def convert_to_uint8(image: np.ndarray, global_max: float = None) -> np.ndarray:
    """
    Convert image to uint8 using a global maximum.
    If global_max is None, it defaults based on dtype (255 for uint8, 2500 for others).
    """
    if global_max is None:
        if image.dtype == np.uint8:
            global_max = 255.0
        else:
            # Heuristic: if max value is low, it's likely uint8 data converted to float
            global_max = 255.0 if np.max(image) <= 255.0 else 2500.0
        
    # Clip values to ensure they stay within the [0, global_max] bounds
    clipped = np.clip(image, 0, global_max)
    
    # Scale to [0, 255] based on the fixed range
    scaled = (clipped / global_max) * 255.0
    
    return scaled.astype(np.uint8)

def apply_transform(source_image: str, transform_params: dict) -> tuple[np.ndarray, any]:

    min_x = transform_params.get("min_x")
    max_x = transform_params.get("max_x")
    min_y = transform_params.get("min_y")
    max_y = transform_params.get("max_y")
    psf_sigma = transform_params.get("psf_sigma")
    snr_db = transform_params.get("snr_db")
    order = transform_params.get("order")
    source_gsd = transform_params.get("source_gsd")
    target_gsd = transform_params.get("target_gsd")

    # extract tile using rasterio        
    tile, geometadata = extract_tile(source_image, min_x, max_x, min_y, max_y)

    # apply transformations
    for op in order:
        if op == "B":
            tile = blur(tile, psf_sigma, source_gsd, target_gsd)
        elif op == "N":
            tile = noise(tile, snr_db)
        elif op == "D":
            tile = downsample(tile, source_gsd, target_gsd)
        else:
            raise ValueError(f"Unknown operation '{op}' in order '{order}'")
    
    return tile, geometadata


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import random


source_dir = "/home/thomas/Documents/code/pleiades-boat-detection/data/raw"
source_csv = "/home/thomas/Documents/code/pleiades-boat-detection/tile_mixture_assignments.csv"
df = pd.read_csv(source_csv) # select random rows for demonstration
# generate random seed
# keep Order "BDN" only

# df = df[df["Transform_id"] == 2]
sample_rows = df.sample(n=1, random_state=random.randint(0, 10000))
# sample_rows = df[:20]

for _, row in sample_rows.iterrows():
    source_image = f"{source_dir}/{row['image_id']}"
    transform_params = {
        "min_x": row["min_x"],
        "max_x": row["max_x"],
        "min_y": row["min_y"],
        "max_y": row["max_y"],
        "psf_sigma": row["PSF"],
        "snr_db": row["SNR (dB)"],
        "order": row["Order"],
        "source_gsd": row["GSD_input"],
        "target_gsd": row["GSD_output"]
    }

    degraded_tile, geometadata = apply_transform(source_image, transform_params)
    print(row)
    original_tile, _ = extract_tile(source_image, transform_params["min_x"], transform_params["max_x"], transform_params["min_y"], transform_params["max_y"])

    # Use the same global_max for both to ensure consistent visualization
    g_max = 255.0 if original_tile.dtype == np.uint8 else 2500.0

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(convert_to_uint8(original_tile[:3], global_max=g_max).transpose(1, 2, 0))
    plt.title("Original Tile")
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(convert_to_uint8(degraded_tile[:3], global_max=g_max).transpose(1, 2, 0))
    plt.title("Degraded Tile")
    plt.axis("off")
    plt.show()



